In [1]:
import json, os, time, logging, joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score


In [2]:
def load_data(data_path, churn_path):
    # Ваш код здесь
    df_features = pd.read_parquet(data_path)
    df_churn = pd.read_csv(churn_path, usecols=['subscriber_key', 'churned'])
    df_churn['churned'] = df_churn['churned'].astype(int)
    df_model = df_features.merge(
        df_churn[['subscriber_key', 'churned']], on='subscriber_key', how='inner'
    )
    df_model['churned'] = df_model['churned'].astype(int)
    if df_model.empty:
        raise ValueError('После объединения признаков и целевой переменной нет строк')
    return df_model
    pass

def preprocess(df_model, seed=42):
    # Ваш код здесь
    split_config = globals().get('config', {}).get('preprocessing', {})
    test_size = float(split_config.get('test_size', 0.4))
    val_size = float(split_config.get('val_size', 0.5))

    df_train, df_temp = train_test_split(
        df_model, test_size=test_size, stratify=df_model['churned'], random_state=seed
    )
    df_val, df_test = train_test_split(
        df_temp, test_size=val_size, stratify=df_temp['churned'], random_state=seed
    )
    feature_cols = (
        df_train.drop(columns=['subscriber_key', 'churned'])
        .select_dtypes(include=[np.number])
        .columns.tolist()
    )
    if not feature_cols:
        raise ValueError('В таблице не найдено числовых признаков')

    X_train_raw = df_train[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    X_val_raw = df_val[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    X_test_raw = df_test[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    y_train = df_train['churned'].to_numpy(dtype=int)
    y_val = df_val['churned'].to_numpy(dtype=int)
    y_test = df_test['churned'].to_numpy(dtype=int)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_val = scaler.transform(X_val_raw)
    X_test = scaler.transform(X_test_raw)
    return X_train, X_val, X_test, y_train, y_val, y_test, feature_cols, scaler
    pass

def train_model(X_train, y_train, params):
    # Ваш код здесь
    model_params = {
        'n_estimators': 100,
        'max_depth': 10,
        'min_samples_split': 2,
        'min_samples_leaf': 1,
        'class_weight': 'balanced',
        'random_state': 42,
        'n_jobs': -1,
    }
    model_params.update(params or {})
    model = RandomForestClassifier(**model_params)
    start_time = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - start_time
    return model, train_time
    pass

def evaluate(model, X_val, y_val, X_test, y_test):
    # Ваш код здесь
    val_proba = model.predict_proba(X_val)[:, 1]
    test_proba = model.predict_proba(X_test)[:, 1]
    val_pred = (val_proba >= 0.5).astype(int)
    test_pred = (test_proba >= 0.5).astype(int)
    metrics = {
        'val_auc': float(roc_auc_score(y_val, val_proba)),
        'val_f1': float(f1_score(y_val, val_pred, zero_division=0)),
        'val_precision': float(precision_score(y_val, val_pred, zero_division=0)),
        'val_recall': float(recall_score(y_val, val_pred, zero_division=0)),
        'test_auc': float(roc_auc_score(y_test, test_proba)),
        'test_f1': float(f1_score(y_test, test_pred, zero_division=0)),
        'test_precision': float(precision_score(y_test, test_pred, zero_division=0)),
        'test_recall': float(recall_score(y_test, test_pred, zero_division=0)),
    }
    return metrics
    pass

def save_artifacts(model, scaler, feature_cols, metrics, config):
    # Ваш код здесь
    model_dir = config['artifacts']['model_dir']
    os.makedirs(model_dir, exist_ok=True)
    paths = {
        'model': os.path.join(model_dir, config['artifacts']['model_name']),
        'scaler': os.path.join(model_dir, config['artifacts']['scaler_name']),
        'features': os.path.join(model_dir, config['artifacts']['features_name']),
        'metrics': os.path.join(model_dir, config['artifacts']['metrics_name']),
    }
    for label, path in paths.items():
        if os.path.isfile(path):
            os.remove(path)
            logging.info('Удалён предыдущий артефакт %s: %s', label, path)

    joblib.dump(model, paths['model'])
    logging.info('Модель сохранена: %s', paths['model'])
    joblib.dump(scaler, paths['scaler'])
    logging.info('Масштабирование сохранено: %s', paths['scaler'])
    with open(paths['features'], 'w', encoding='utf-8') as f:
        json.dump(feature_cols, f, indent=2, ensure_ascii=False)
    logging.info('Признаки сохранены: %s', paths['features'])
    with open(paths['metrics'], 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    logging.info('Метрики сохранены: %s', paths['metrics'])
    print(f'Артефакты сохранены идемпотентно в {model_dir}')
    pass

def load_data_safe(data_path, churn_path):
    # Ваш код здесь
    try:
        for path in (data_path, churn_path):
            if not os.path.isfile(path):
                raise FileNotFoundError(f'Не найден файл: {path}')
        return load_data(data_path, churn_path)
    except Exception:
        logging.exception('Ошибка загрузки данных')
        raise
    pass

def run_pipeline(config_path):
    # Ваш код здесь
    with open(config_path, 'r', encoding='utf-8') as f:
        pipeline_config = json.load(f)
    # Используем настройки именно той конфигурации, с которой запущен конвейер.
    globals()['config'] = pipeline_config
    logging.info('Шаг 1/5: загрузка данных')
    df_model = load_data_safe(
        pipeline_config['data']['features_path'],
        pipeline_config['data']['churn_path'],
    )
    logging.info('Загружено строк: %d', len(df_model))

    logging.info('Шаг 2/5: предобработка и стратифицированное разделение')
    X_train, X_val, X_test, y_train, y_val, y_test, feature_cols, scaler = preprocess(
        df_model, seed=pipeline_config['preprocessing']['random_state']
    )
    logging.info('Число признаков: %d', len(feature_cols))

    logging.info('Шаг 3/5: обучение Random Forest')
    model, train_time = train_model(X_train, y_train, pipeline_config['model'])
    logging.info('Обучение завершено за %.2f сек', train_time)

    logging.info('Шаг 4/5: оценка на Validation и Test')
    metrics = evaluate(model, X_val, y_val, X_test, y_test)
    metrics['train_time_sec'] = float(train_time)
    logging.info('Test ROC-AUC: %.4f', metrics['test_auc'])

    logging.info('Шаг 5/5: сохранение артефактов')
    save_artifacts(model, scaler, feature_cols, metrics, pipeline_config)
    logging.info('Конвейер выполнен успешно')
    return model, metrics
    pass

In [3]:
config_path = '/content/project_6.1/config.json'
with open(config_path, encoding='utf-8') as f:
    config = json.load(f)
logging.basicConfig(level=logging.INFO,
                    format=config['logging']['format'],
                    handlers=[logging.FileHandler('/content/project_6.1/pipeline.log', mode='w'),
                              logging.StreamHandler()], force=True)
model, metrics = run_pipeline(config_path)
print('Test ROC-AUC:', round(metrics['test_auc'], 4))


2026-11-20 10:48:54,263 - INFO - Шаг 1/5: загрузка данных


2026-11-20 10:48:54,318 - INFO - Загружено строк: 400


2026-11-20 10:48:54,320 - INFO - Шаг 2/5: предобработка и стратифицированное разделение


2026-11-20 10:48:54,358 - INFO - Число признаков: 2


2026-11-20 10:48:54,360 - INFO - Шаг 3/5: обучение Random Forest


2026-11-20 10:48:54,687 - INFO - Обучение завершено за 0.32 сек


... (5 outputs left out)

2026-11-20 10:48:54,869 - INFO - Признаки сохранены: /content/processed/model_6.1/features.json


2026-11-20 10:48:54,872 - INFO - Метрики сохранены: /content/processed/model_6.1/metrics.json


2026-11-20 10:48:54,873 - INFO - Конвейер выполнен успешно


Артефакты сохранены идемпотентно в /content/processed/model_6.1
Test ROC-AUC: 0.8649
